In [9]:
import sys, os
import nibabel as nib
from tqdm import tqdm
from pathlib import Path

repo_path = os.path.abspath('~/ukbb-pulmonary-artery/DeepCMR')
assert os.path.isdir(repo_path)
if not repo_path in sys.path: sys.path.append(repo_path)

from utils.strings import str2
import utils.strings as dstr

### Define folders

In [5]:
# ================================== EDIT THESE ==================================
# Location of our project folder
PROJ_ROOT = '{deepcmr_data_root}/nnUNET/'

# Training and test data subfolders
training_niftis = '../OrthancDicomStorageNiftis/'
testing_niftis = '../OrthancDicomStorage-PracticeNiftis/'

# Task name (for nnUNet)
task_name = 'Task618_UKBBPulmonaryArtery'
# ================================================================================

In [ ]:
# nnUNET folder structure, including task name 
assert task_name.startswith('Task')
assert task_name.split('_')[0].split('Task')[1].isnumeric()
nnUNet_data_path = os.path.join(PROJ_ROOT, 'nnUNet_raw_data_base/nnUNet_raw_data/', task_name + '/')
dataset_json_path = nnUNet_data_path + 'dataset.json'

os.makedirs(nnUNet_data_path, exist_ok=True)

# Create list of Niftis found in the data folder
FileNames = [ str2(name).rchop('.nii.gz')
                    for i in Path(PROJ_ROOT, training_niftis).glob('*.nii.gz') if not str(i).endswith('_gt.nii.gz') ]
FileNames_testing = [ str2(name).rchop('.nii.gz')
                            for i in Path(PROJ_ROOT, testing_niftis).glob('*.nii.gz') if not str(i).endswith('_gt.nii.gz') ]

print("Number of training set patients: %.d" % len(FileNames))
print("Number of testing set patients: %.d" % len(FileNames_testing))

## Generate training data files

In [19]:
def extract_nifts_and_get_labels(fileNames, outputFolder, **kwargs):
    test = kwargs.pop('test', False)

    folder_suffix = 'Ts' if test else 'Tr'

    i = 0
    d_labels = []
    for filename in tqdm(fileNames, desc="Progress", unit="patient"):
        V_nifti = nib.load(filename + '.nii.gz')
        M_nifti = nib.load(filename + '_gt.nii.gz')
        
        affine = V_nifti.affine
        V = V_nifti.get_fdata()
        M = M_nifti.get_fdata()
        for t in range(50):
            V_nifti_t = nib.Nifti1Image(V[:,:,:,t], affine)
            M_nifti_t = nib.Nifti1Image(M[:,:,:,t], affine)
            
            os.makedirs(os.path.join(outputFolder, 'images' + folder_suffix), exist_ok=True)
            os.makedirs(os.path.join(outputFolder, 'labels' + folder_suffix), exist_ok=True)
            
            # unique filename, including patient name and frame number.
            # - remove problematic characters
            # - nn-unet wants there to be a unique numeric identifier at the end.
            shortname = dstr.standardize_patient_name(str2(filename.basename()))
            file_prefix = dstr.format_patient_filename(prefix='pa', patient_name=shortname, frame_num=t, index=i)

            image_path = os.path.join(outputFolder, 'images' + folder_suffix, file_prefix + '_0000.nii.gz')
            label_path = os.path.join(outputFolder, 'labels' + folder_suffix, file_prefix + '.nii.gz')

            V_nifti_t.to_filename(image_path)
            M_nifti_t.to_filename(label_path)
            
            d_labels += [file_prefix]
            i += 1
    print('')
    return d_labels

In [ ]:
dtrn_labels = extract_nifts_and_get_labels(FileNames, nnUNet_data_path, test=False)
dtst_labels = extract_nifts_and_get_labels(FileNames_testing, nnUNet_data_path, test=True)

print("Number of training set labels: %.d" % len(dtrn_labels))
print("Number of testing set labels: %.d" % len(dtst_labels))

### Create nnUNet json dataset file

In [21]:
def write_nnUNet_dataset_json(fname, **kwargs):
    
    name = kwargs.pop('name', 'UKBBPulmonaryArtery')
    description = kwargs.pop('description', 'Pulmonary Artery segmentation')
    reference = kwargs.pop('reference', 'UK Biobank')
    tensorImageSize = kwargs.pop('tensorImageSize', '4D')
    modalities = kwargs.pop('modalities', ['MRI'])
    labels = kwargs.pop('labels', ['background', 'PA'])
    dtrns = kwargs.pop('dtrns', ['pa_00000','pa_00001','pa_00002'])
    dtsts = kwargs.pop('dtsts', [])
    
   
    with open(fname, 'w') as f:
        f.write('{\n')
        f.write(""""name":"%s",\n"""%(name))
        f.write(""""description":"%s",\n"""%(description))
        f.write(""""reference":"%s",\n"""%(reference))
        f.write(""""tensorImageSize":"%s",\n"""%(tensorImageSize))
        
        f.write(""""modality":{\n""")
        for j, modality in enumerate(modalities):
            if j == len(modalities) - 1:
                f.write(""""%d":"%s"\n"""%(j, modality))
            else:
                f.write(""""%d":"%s",\n"""%(j, modality))
                
        f.write("""},\n""")
        
        f.write(""""labels":{\n""")
        for j, label in enumerate(labels):
            if j == len(label) - 1:
                f.write(""""%d":"%s"\n"""%(j, label))
            else:
                f.write(""""%d":"%s",\n"""%(j, label))
        f.write("""},\n""")
        
        f.write(""""numTraining":%d,\n"""%(len(dtrns)))
        f.write(""""numTest":%d,\n"""%(len(dtsts)))
        
        samples = ["""{"image":"./imagesTr/%s.nii.gz","label":"./labelsTr/%s.nii.gz"}"""%(dtrn,dtrn) for dtrn in dtrns]
        
        f.write(""""training":[\n""")
        for j, sample in enumerate(samples):
            if j == len(samples) - 1:
                f.write("""%s\n"""%(sample))
            else:
                f.write("""%s,\n"""%(sample))
        f.write("""],\n""")
          
        if len(dtsts) > 0:
            samples = [""""./imagesTs/%s.nii.gz" """%(dtst) for dtst in dtsts]

            f.write(""""test":[\n""")
            for j, sample in enumerate(samples):
                if j == len(samples) - 1:
                    f.write("""%s\n"""%(sample))
                else:
                    f.write("""%s,\n"""%(sample))
            f.write("""]\n""")
        else:
            f.write(""""test":[]\n""")
        
        f.write("""}""")

In [22]:
write_nnUNet_dataset_json(dataset_json_path, dtrns=dtrn_labels, dtsts=dtst_labels)